In [1]:
# import packages 
import numpy as np
import pandas as pd
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

df = pd.read_csv('../data/cleaned_data.csv')

In [2]:
df['start_time'] = pd.to_datetime(df['start_time'])

# Create features from datetime
df['month'] = df['start_time'].dt.month
df['dayofweek'] = df['start_time'].dt.dayofweek
df['day'] = df['start_time'].dt.day

# One-hot encode categorical variables
X = pd.get_dummies(df[['state', 'Event Type', 'county', 'duration',
                       'month', 'dayofweek', 'day']], drop_first=True)
y = df['hour']

# --- Randomly sample 2000 rows ---
sample_size = 2000
df_small = df.sample(n=sample_size, random_state=42)  # keeps reproducibility

# Split features and target for the small sample
X_small = pd.get_dummies(df_small[['state', 'Event Type', 'county', 'duration',
                                   'month', 'dayofweek', 'day']], drop_first=True)
y_small = df_small['hour']

print("Shape of X_small:", X_small.shape)
print("Shape of y_small:", y_small.shape)

Shape of X_small: (2000, 564)
Shape of y_small: (2000,)


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Parameter distribution
lr_param_dist = {
    'fit_intercept': [True, False],
    'positive': [True, False]
}

# Initialize model
lr = LinearRegression()

# Randomized Search
lr_rand = RandomizedSearchCV(
    lr,
    lr_param_dist,
    n_iter=10,      # number of random combinations to try
    cv=5,           # fewer folds for speed on small dataset
    scoring='r2',
    n_jobs=-1,
    random_state=42,
)

# Fit the model on small training set
lr_rand.fit(X_train_scaled, y_train)

# Predictions
y_pred_lr_tuned = lr_rand.best_estimator_.predict(X_test_scaled)

# Metrics
print("Test R2:", r2_score(y_test, y_pred_lr_tuned))
print("MAE:", mean_absolute_error(y_test, y_pred_lr_tuned))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lr_tuned)))

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# --- Use your small sample ---
X = X_small  # 2000-row random subset of features
y = y_small  # corresponding target

# Split into train/test for the small sample
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features if needed (optional for Random Forest)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Random Forest hyperparameter grid ---
rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

# Initialize model
rf = RandomForestRegressor(random_state=42)

# Randomized Search (note: n_iter=4 matches total combinations)
rf_random = RandomizedSearchCV(
    rf,
    rf_param_grid,
    n_iter=4,      # number of random combinations
    cv=3,          # fewer folds for speed
    scoring='r2',
    n_jobs=-1,     # parallelize
    random_state=42
)

# Fit the model on the small training set
rf_random.fit(X_train_scaled, y_train)

# Predict on the small test set
y_pred_rf_tuned = rf_random.best_estimator_.predict(X_test_scaled)

# Metrics
print("Random Forest Results on Small Sample")
print("Test R2:", r2_score(y_test, y_pred_rf_tuned))
print("MAE:", mean_absolute_error(y_test, y_pred_rf_tuned))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_rf_tuned)))

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

# Hyperparameter grid
xgb_param_grid = {
    'n_estimators': [200, 400],
    'learning_rate': [0.05, 0.1],
    'max_depth': [4, 6, 8],
    'subsample': [0.7, 0.9],
    'colsample_bytree': [0.7, 1.0]
}

# Initialize model
xgb = XGBRegressor(random_state=42, n_jobs=-1)

# Randomized Search
xgb_random = RandomizedSearchCV(
    xgb,
    xgb_param_grid,
    n_iter=4,   # small sample, keep low
    cv=3,
    scoring='r2',
    n_jobs=-1,
    random_state=42
)

# Fit the model on the small sample
xgb_random.fit(X_train, y_train)

# Predict
y_pred_xgb_tuned = xgb_random.best_estimator_.predict(X_test)

# Metrics
r2 = r2_score(y_test, y_pred_xgb_tuned)
mae = mean_absolute_error(y_test, y_pred_xgb_tuned)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_xgb_tuned))

print("XGBoost Results on Small Sample")
print("Test R2:", r2)
print("MAE (Hours):", mae)
print("RMSE (Hours):", rmse)